# HW02 — MLflow Experiment Tracking

In HW01, you built a versioned feature dataset for the Airbnb listing availability problem.

In this notebook, you will train several model versions and track them in MLflow.

The goal is not only to get a high score. The goal is to make every experiment reproducible:

- which dataset version was used
- which features were used
- which model was trained
- which parameters were used
- which metrics were produced
- which artifacts were saved
- which run should be considered the final candidate

MLflow server:

```text
http://185.50.38.163:33014
```

Use your assigned MLflow username/password and your assigned experiment name from the credentials sheet.

## Required output

By the end of this notebook, you must have:

1. At least **5 MLflow runs**.
2. At least **3 different experiment types**:
   - one intentionally leaky run
   - one baseline run
   - at least one clean real model
3. Logged parameters, metrics, tags, artifacts, and an sklearn Pipeline model.
4. A run comparison table.
5. One selected final candidate run.
6. A short explanation of why that run was selected.

Do not use future/label columns in your final clean model.

In [5]:
# If needed, install these in your local environment first:
!pip install pandas numpy scikit-learn matplotlib mlflow pyarrow

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

import mlflow
import mlflow.sklearn

RANDOM_STATE = 42

Defaulting to user installation because normal site-packages is not writeable


## 1. Configure MLflow

Fill in your assigned MLflow credentials.

Important:

- `MLFLOW_TRACKING_URI` is the shared MLflow server.
- `MLFLOW_USERNAME` and `MLFLOW_PASSWORD` are **not** your database credentials.
- `EXPERIMENT_NAME` must be your own assigned experiment name, for example:

```text
qbc12_hw02_student_nazanin_hesari
```

Do not use someone else's experiment name.

In [3]:
!pip install mlflow scikit-learn lightgbm pandas numpy dvc

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
MLFLOW_TRACKING_URI = "http://185.50.38.163:33014"

# TODO: replace these with your assigned MLflow credentials.
MLFLOW_USERNAME = "student_your_username"
MLFLOW_PASSWORD = "your_mlflow_password"
EXPERIMENT_NAME = "qbc12_hw02_student_your_username"

if MLFLOW_USERNAME == "student_your_username" or MLFLOW_PASSWORD == "your_mlflow_password":
    raise ValueError("Replace MLFLOW_USERNAME, MLFLOW_PASSWORD, and EXPERIMENT_NAME with your assigned values.")

os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", experiment.name if experiment else None)
print("Experiment ID:", experiment.experiment_id if experiment else None)

In [ ]:
import os
from dotenv import load_dotenv
import mlflow

load_dotenv()

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
MLFLOW_USERNAME = os.getenv("MLFLOW_TRACKING_USERNAME")
MLFLOW_PASSWORD = os.getenv("MLFLOW_TRACKING_PASSWORD")
EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME")
STUDENT_NAME = os.getenv("STUDENT_NAME", "unknown_student")

required_envs = {
    "MLFLOW_TRACKING_URI": MLFLOW_TRACKING_URI,
    "MLFLOW_TRACKING_USERNAME": MLFLOW_USERNAME,
    "MLFLOW_TRACKING_PASSWORD": MLFLOW_PASSWORD,
    "MLFLOW_EXPERIMENT_NAME": EXPERIMENT_NAME,
}

missing = [key for key, value in required_envs.items() if not value]

if missing:
    raise ValueError(
        "Missing required environment variables: "
        + ", ".join(missing)
        + ". Create a .env file or export them in your shell."
    )

os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", mlflow.get_experiment_by_name(EXPERIMENT_NAME))

## 2. Load the HW01 dataset

Use the cleaned dataset produced by HW01.

Expected files:

```text
data/features/listing_availability_features_v1_audit_cleaned.csv
data/features/listing_availability_features_v1_audit_cleaned.parquet
data/features/listing_availability_features_v1_audit_cleaned_metadata.json
```

You may use CSV or Parquet. Parquet is preferred if available.

In [13]:
import pandas as pd


DATA_PATH = "listing_availability_features_v1_student.csv"  

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.dtypes)
print("\nFirst rows:")
df.head()

Shape: (10480, 32)

Columns:
listing_id                               int64
room_type                               object
property_type                           object
accommodates                             int64
bedrooms                               float64
beds                                   float64
bathrooms                              float64
listing_price                          float64
minimum_nights                           int64
maximum_nights                           int64
instant_bookable                          bool
is_superhost                            object
host_listing_count                       int64
neighbourhood_name                      object
total_reviews_before_cutoff            float64
unique_reviewers_before_cutoff         float64
avg_comment_len_before_cutoff          float64
max_comment_len_before_cutoff          float64
days_since_last_review                 float64
available_days_history                   int64
available_rate_history         

,listing_id,room_type,property_type,accommodates,bedrooms,beds,bathrooms,listing_price,minimum_nights,maximum_nights,...,avg_maximum_nights_calendar_history,available_days_last_30d,available_rate_last_30d,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy,cutoff_date,dataset_version,has_reviews
0,27886,Private room,Private room in houseboat,2,1.0,1.0,1.5,132.0,3,356,...,30.0,0,0.000000,30,0,0.0,1,2026-08-11,v1_student,1
1,28871,Private room,Private room in rental unit,2,1.0,1.0,1.0,89.0,2,730,...,730.0,14,0.466667,30,21,0.7,0,2026-08-11,v1_student,1
2,29051,Private room,Private room in condo,2,1.0,1.0,1.0,61.0,2,730,...,730.0,16,0.533333,30,0,0.0,1,2026-08-11,v1_student,1
3,44391,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5,NaN,3,730,...,730.0,0,0.000000,30,0,0.0,1,2026-08-11,v1_student,1
4,48373,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5,NaN,3,1125,...,1125.0,0,0.000000,30,0,0.0,1,2026-08-11,v1_student,1


In [15]:
DATA_PATH = "listing_availability_features_v1_student.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())
display(df.dtypes)

Shape: (10480, 32)


,listing_id,room_type,property_type,accommodates,bedrooms,beds,bathrooms,listing_price,minimum_nights,maximum_nights,...,avg_maximum_nights_calendar_history,available_days_last_30d,available_rate_last_30d,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy,cutoff_date,dataset_version,has_reviews
0,27886,Private room,Private room in houseboat,2,1.0,1.0,1.5,132.0,3,356,...,30.0,0,0.000000,30,0,0.0,1,2026-08-11,v1_student,1
1,28871,Private room,Private room in rental unit,2,1.0,1.0,1.0,89.0,2,730,...,730.0,14,0.466667,30,21,0.7,0,2026-08-11,v1_student,1
2,29051,Private room,Private room in condo,2,1.0,1.0,1.0,61.0,2,730,...,730.0,16,0.533333,30,0,0.0,1,2026-08-11,v1_student,1
3,44391,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5,NaN,3,730,...,730.0,0,0.000000,30,0,0.0,1,2026-08-11,v1_student,1
4,48373,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5,NaN,3,1125,...,1125.0,0,0.000000,30,0,0.0,1,2026-08-11,v1_student,1


listing_id                               int64
room_type                               object
property_type                           object
accommodates                             int64
bedrooms                               float64
beds                                   float64
bathrooms                              float64
listing_price                          float64
minimum_nights                           int64
maximum_nights                           int64
instant_bookable                          bool
is_superhost                            object
host_listing_count                       int64
neighbourhood_name                      object
total_reviews_before_cutoff            float64
unique_reviewers_before_cutoff         float64
avg_comment_len_before_cutoff          float64
max_comment_len_before_cutoff          float64
days_since_last_review                 float64
available_days_history                   int64
available_rate_history                 float64
avg_minimum_n

In [17]:
TARGET = "high_demand_proxy"

print("Target counts:")
display(df[TARGET].value_counts())

print("Target ratio:")
display(df[TARGET].value_counts(normalize=True))

Target counts:


high_demand_proxy
1    7994
0    2486
Name: count, dtype: int64

Target ratio:


high_demand_proxy
1    0.762786
0    0.237214
Name: proportion, dtype: float64

In [19]:
LEAKY_COLS = [
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
]

META_COLS = [
    "listing_id",
    "cutoff_date",
    "dataset_version",
]

CATEGORICAL_COLS = [
    "room_type",
    "property_type",
    "is_superhost",
    "neighbourhood_name",
]

print("Leaky columns:", LEAKY_COLS)
print("Meta columns:", META_COLS)

Leaky columns: ['future_calendar_days_observed_30d', 'future_available_days_30d', 'future_available_rate_30d']
Meta columns: ['listing_id', 'cutoff_date', 'dataset_version']


In [21]:
print("Mean future availability by target:")
display(
    df.groupby(TARGET)[
        [
            "future_available_days_30d",
            "future_available_rate_30d",
            "available_days_history",
            "available_rate_history",
            "available_days_last_30d",
            "available_rate_last_30d",
        ]
    ].mean()
)

print("Correlation with target:")
numeric_corr = df.select_dtypes(include=[np.number]).corr(numeric_only=True)[TARGET].sort_values(ascending=False)
display(numeric_corr)

Mean future availability by target:


,future_available_days_30d,future_available_rate_30d,available_days_history,available_rate_history,available_days_last_30d,available_rate_last_30d
high_demand_proxy,,,,,,
0,28.699115,0.956637,80.081255,0.880014,27.112631,0.903754
1,0.041656,0.001389,3.224418,0.035433,0.248561,0.008285


Correlation with target:


high_demand_proxy                      1.000000
days_since_last_review                 0.174857
has_reviews                            0.067982
bedrooms                               0.032331
bathrooms                              0.007987
avg_maximum_nights_calendar_history    0.007704
avg_comment_len_before_cutoff          0.003289
minimum_nights                        -0.017824
avg_minimum_nights_calendar_history   -0.021328
listing_price                         -0.023160
accommodates                          -0.027000
beds                                  -0.038010
host_listing_count                    -0.071409
listing_id                            -0.075936
maximum_nights                        -0.080229
max_comment_len_before_cutoff         -0.093266
total_reviews_before_cutoff           -0.130254
unique_reviewers_before_cutoff        -0.130271
available_days_history                -0.917913
available_rate_history                -0.917913
available_days_last_30d               -0

In [ ]:
DATASET_VERSION = "v1_audit_cleaned"

FEATURE_DIR = Path("data/features")

parquet_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.parquet"
csv_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.csv"
metadata_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_metadata.json"

# TODO: load the dataset.
# Prefer Parquet if it exists, otherwise use CSV.
raise NotImplementedError("Load feature_df from parquet_path or csv_path.")

# TODO: load metadata if metadata_path exists.
metadata = {}

print(feature_df.shape)
feature_df.head()

## 3. Define target and forbidden columns

The target is:

```text
high_demand_proxy
```

The following columns must **not** be used as clean model inputs:

```text
listing_id
cutoff_date
dataset_version
future_calendar_days_observed_30d
future_available_days_30d
future_available_rate_30d
high_demand_proxy
```

Why?

- `high_demand_proxy` is the label.
- `future_*` columns are from the label window.
- `listing_id`, `cutoff_date`, and `dataset_version` are audit/entity fields, not predictive features.

You will intentionally use one future column in the **leaky run only** to show what leakage looks like. Your final model must be clean.

In [ ]:
TARGET_COL = "high_demand_proxy"

FORBIDDEN_MODEL_COLUMNS = [
    "listing_id",
    "cutoff_date",
    "dataset_version",
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
]

# TODO: check that TARGET_COL exists.
# TODO: create y.
# TODO: create clean feature list by excluding FORBIDDEN_MODEL_COLUMNS.
# TODO: create X_clean.

raise NotImplementedError("Create y, clean_feature_cols, and X_clean.")

print("Target distribution:")
print(y.value_counts(normalize=True).sort_index())

print("Clean feature count:", len(clean_feature_cols))
print(clean_feature_cols)

In [23]:
def make_features(df, drop_leaky=True):
    data = df.copy()

    # تبدیل bool به int
    if "instant_bookable" in data.columns:
        data["instant_bookable"] = data["instant_bookable"].astype(int)

    # تبدیل is_superhost
    if "is_superhost" in data.columns:
        data["is_superhost"] = (
            data["is_superhost"]
            .astype(str)
            .str.lower()
            .map({
                "t": 1,
                "true": 1,
                "1": 1,
                "f": 0,
                "false": 0,
                "0": 0,
                "nan": 0,
                "none": 0
            })
            .fillna(0)
            .astype(int)
        )

    drop_cols = [TARGET] + META_COLS

    if drop_leaky:
        drop_cols += LEAKY_COLS

    drop_cols = [c for c in drop_cols if c in data.columns]

    X = data.drop(columns=drop_cols)
    y = data[TARGET].astype(int)

    # One-hot encoding
    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

    # پر کردن missing values
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True))

    return X, y

## 4. Create one intentionally leaky feature set

This run is supposed to be wrong.

Create `X_leaky` by allowing `future_available_rate_30d` into the features.

The point is to show that a model can look excellent for the wrong reason. Log this run with:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
```

Do not select this run as your final model.

In [25]:
X_clean, y_clean = make_features(df, drop_leaky=True)
X_leaky, y_leaky = make_features(df, drop_leaky=False)

print("Clean feature shape:", X_clean.shape)
print("Leaky feature shape:", X_leaky.shape)
print("Target shape:", y_clean.shape)

Clean feature shape: (10480, 108)
Leaky feature shape: (10480, 111)
Target shape: (10480,)


In [ ]:
LEAKAGE_COLUMN = "future_available_rate_30d"

# TODO: create leaky_feature_cols.
# It should include the clean features plus LEAKAGE_COLUMN.
# It must still exclude the target itself.

raise NotImplementedError("Create leaky_feature_cols and X_leaky.")

print("Leaky feature count:", len(leaky_feature_cols))
print("Leakage column included:", LEAKAGE_COLUMN in leaky_feature_cols)

## 5. Train/test split

Use a stratified split.

Why stratified?

The target is not perfectly balanced, so the train and test sets should preserve the class ratio.

In [ ]:
# TODO: split X_clean and y.
# Use test_size=0.20, random_state=42, stratify=y.

raise NotImplementedError("Create X_train, X_test, y_train, y_test.")

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target rate:", y_train.mean())
print("Test target rate:", y_test.mean())

In [27]:
def get_dataframe_hash(df):
    hashed = pd.util.hash_pandas_object(df, index=True).values
    return hashlib.md5(hashed).hexdigest()[:12]

DATASET_HASH = get_dataframe_hash(df)
DATASET_VERSION = df["dataset_version"].iloc[0] if "dataset_version" in df.columns else "unknown"

print("Dataset hash:", DATASET_HASH)
print("Dataset version:", DATASET_VERSION)

Dataset hash: a1c4af14e0dd
Dataset version: v1_student


## 6. Build preprocessing

Use an sklearn `ColumnTransformer`.

Required preprocessing:

- numeric columns:
  - median imputation
  - standard scaling
- categorical columns:
  - most-frequent imputation
  - one-hot encoding

The logged model must be a full sklearn `Pipeline`, not just the estimator.

In [ ]:
def make_one_hot_encoder():
    """Return OneHotEncoder compatible with multiple sklearn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


# TODO: identify numeric_cols and categorical_cols from X_clean.
# Hint: numeric columns usually have dtype int/float.
# Everything else can be treated as categorical.

raise NotImplementedError("Create numeric_cols and categorical_cols.")

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_cols),
        ("categorical", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
)

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

In [29]:
def log_confusion_matrix_artifact(y_true, y_pred, filename="confusion_matrix.txt"):
    cm = confusion_matrix(y_true, y_pred)
    
    with open(filename, "w") as f:
        f.write("Confusion Matrix\n")
        f.write(str(cm))
        f.write("\n\nClassification Report\n")
        f.write(classification_report(y_true, y_pred))
    
    mlflow.log_artifact(filename)

## 7. Evaluation helpers

Complete the evaluation helper.

Every run must log the same metric set:

```text
accuracy
precision
recall
f1
roc_auc
```

Use `zero_division=0` for precision/recall/f1.

In [ ]:
def get_positive_scores(model, X):
    """Return positive-class scores for binary classifiers."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        raw = model.decision_function(X)
        return 1 / (1 + np.exp(-raw))
    return model.predict(X)


def evaluate_binary_classifier(model, X_test, y_test, threshold=0.5):
    """Evaluate a fitted binary classifier."""
    # TODO:
    # 1. get positive scores
    # 2. convert scores to predictions using threshold
    # 3. calculate accuracy, precision, recall, f1, roc_auc
    # 4. return metrics dict, y_pred, y_score

    raise NotImplementedError("Complete evaluate_binary_classifier.")

In [31]:
def run_experiment(
    run_name,
    model,
    X_train,
    X_test,
    y_train,
    y_test,
    params,
    leaky,
    description
):
    with mlflow.start_run(run_name=run_name):
        
        # -----------------------
        # Params
        # -----------------------
        mlflow.log_param("dataset_path", DATA_PATH)
        mlflow.log_param("dataset_hash", DATASET_HASH)
        mlflow.log_param("dataset_version", DATASET_VERSION)
        mlflow.log_param("n_rows", df.shape[0])
        mlflow.log_param("n_raw_columns", df.shape[1])
        mlflow.log_param("n_features", X_train.shape[1])
        mlflow.log_param("target", TARGET)
        mlflow.log_param("leaky_features_used", leaky)
        mlflow.log_params(params)

        # -----------------------
        # Tags
        # -----------------------
        mlflow.set_tag("description", description)
        mlflow.set_tag("problem_type", "binary_classification")
        mlflow.set_tag("student", "sobhan_jabari")
        
        if leaky:
            mlflow.set_tag("valid_model", "false")
            mlflow.set_tag("reason", "uses future columns, leakage")
        else:
            mlflow.set_tag("valid_model", "true")

        # -----------------------
        # Train
        # -----------------------
        model.fit(X_train, y_train)

        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)

        # predict_proba برای AUC
        if hasattr(model, "predict_proba"):
            y_proba_train = model.predict_proba(X_train)[:, 1]
            y_proba_test = model.predict_proba(X_test)[:, 1]
        else:
            y_proba_train = y_pred_train
            y_proba_test = y_pred_test

        # -----------------------
        # Metrics
        # -----------------------
        metrics = {
            "train_accuracy": accuracy_score(y_train, y_pred_train),
            "test_accuracy": accuracy_score(y_test, y_pred_test),
            "train_precision": precision_score(y_train, y_pred_train, zero_division=0),
            "test_precision": precision_score(y_test, y_pred_test, zero_division=0),
            "train_recall": recall_score(y_train, y_pred_train, zero_division=0),
            "test_recall": recall_score(y_test, y_pred_test, zero_division=0),
            "train_f1": f1_score(y_train, y_pred_train, zero_division=0),
            "test_f1": f1_score(y_test, y_pred_test, zero_division=0),
            "train_auc": roc_auc_score(y_train, y_proba_train),
            "test_auc": roc_auc_score(y_test, y_proba_test),
        }

        metrics["auc_gap"] = metrics["train_auc"] - metrics["test_auc"]
        metrics["f1_gap"] = metrics["train_f1"] - metrics["test_f1"]

        mlflow.log_metrics(metrics)

        # -----------------------
        # Artifacts
        # -----------------------
        log_confusion_matrix_artifact(y_test, y_pred_test)

        # Feature importance برای مدل‌های tree-based
        if hasattr(model, "feature_importances_"):
            fi = pd.Series(model.feature_importances_, index=X_train.columns)
            fi = fi.sort_values(ascending=False)

            fi.to_csv("feature_importance.csv")
            mlflow.log_artifact("feature_importance.csv")

            print(f"\nTop 10 features for {run_name}:")
            display(fi.head(10))

        # -----------------------
        # Model
        # -----------------------
        signature = infer_signature(X_train, model.predict(X_train))

        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model",
            signature=signature,
            input_example=X_train.head(3)
        )

        print("=" * 80)
        print(run_name)
        print("=" * 80)
        for k, v in metrics.items():
            print(f"{k}: {v:.4f}")

        return metrics

## 8. Artifact helpers

Each serious run should save useful artifacts:

- confusion matrix image
- classification report JSON
- feature column list JSON
- dataset metadata snapshot JSON

Artifacts are important because MLflow should store more than scalar metrics.

In [ ]:
ARTIFACT_DIR = Path("outputs/mlflow_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def save_run_artifacts(run_name, y_true, y_pred, feature_cols, metadata):
    """Save local artifact files for one run and return the run artifact directory."""
    # TODO:
    # 1. create a run-specific artifact folder
    # 2. save confusion_matrix.png
    # 3. save classification_report.json
    # 4. save feature_columns.json
    # 5. save dataset_metadata_snapshot.json

    raise NotImplementedError("Complete save_run_artifacts.")

## 9. MLflow run helper

Complete a helper that:

1. fits the pipeline,
2. evaluates it,
3. logs params,
4. logs metrics,
5. logs tags,
6. logs artifacts,
7. logs the full sklearn Pipeline model.

Use the same helper for all model versions. That is the point of experiment tracking.

In [ ]:
def run_mlflow_experiment(
    run_name,
    pipeline,
    X_train,
    X_test,
    y_train,
    y_test,
    feature_cols,
    model_params,
    tags,
    threshold=0.5,
):
    # TODO: implement this function.
    # Required MLflow calls:
    # - mlflow.start_run(run_name=run_name)
    # - mlflow.log_params(...)
    # - mlflow.log_metrics(...)
    # - mlflow.set_tags(...)
    # - mlflow.log_artifacts(...)
    # - mlflow.sklearn.log_model(...)

    raise NotImplementedError("Complete run_mlflow_experiment.")

## 10. Run 0 — intentionally leaky model

This run is wrong on purpose.

Use a real model, but include `future_available_rate_30d`.

Expected behavior: performance may look suspiciously strong.

Required tags:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
model_family = logistic_regression
```

In [ ]:
# TODO:
# 1. split X_leaky and y using the same stratified split settings
# 2. build a LogisticRegression pipeline
# 3. log the run to MLflow

raise NotImplementedError("Run and log v0_leaky_logistic_regression.")

In [43]:
from mlflow.models.signature import infer_signature

In [45]:
RANDOM_STATE = 42
TEST_SIZE = 0.2

# دیتاست leaky برای مدل نمایشی بد
X_leaky, y_leaky = make_features(df, drop_leaky=False)

X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(
    X_leaky,
    y_leaky,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_leaky
)

# دیتاست clean برای مدل‌های واقعی
X_clean, y_clean = make_features(df, drop_leaky=True)

X_train_clean, X_test_clean, y_train_clean, y_test_clean = train_test_split(
    X_clean,
    y_clean,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_clean
)

print("Leaky train/test:", X_train_leaky.shape, X_test_leaky.shape)
print("Clean train/test:", X_train_clean.shape, X_test_clean.shape)

Leaky train/test: (8384, 111) (2096, 111)
Clean train/test: (8384, 108) (2096, 108)


## 11. Run 1 — dummy baseline

Train a `DummyClassifier(strategy="most_frequent")`.

This tells you what a useless model can achieve.

If your real model barely beats this, your model is weak.

In [ ]:
# TODO: build and log dummy baseline.

raise NotImplementedError("Run and log v1_dummy_baseline.")

In [51]:
import os
import shutil
import tempfile
import pickle

import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np

from mlflow.models.signature import infer_signature

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


def log_confusion_matrix_artifact(y_true, y_pred, filename="confusion_matrix.txt"):
    cm = confusion_matrix(y_true, y_pred)
    
    with open(filename, "w", encoding="utf-8") as f:
        f.write("Confusion Matrix\n")
        f.write(str(cm))
        f.write("\n\nClassification Report\n")
        f.write(classification_report(y_true, y_pred, zero_division=0))
    
    mlflow.log_artifact(filename)


def save_and_log_model_as_artifact(model, X_train, artifact_path="model"):
    """
    Compatible workaround for MLflow servers that do not support /logged-models endpoint.
    Saves sklearn model locally, then logs the model folder as artifacts.
    """
    tmp_dir = tempfile.mkdtemp()
    model_dir = os.path.join(tmp_dir, artifact_path)

    try:
        # Signature
        signature = infer_signature(X_train, model.predict(X_train))

        # Save MLflow sklearn model locally
        mlflow.sklearn.save_model(
            sk_model=model,
            path=model_dir,
            signature=signature,
            input_example=X_train.head(3)
        )

        # Log the saved model directory as artifact
        mlflow.log_artifacts(model_dir, artifact_path=artifact_path)

        # Also save simple pickle backup
        pickle_path = os.path.join(tmp_dir, "model.pkl")
        with open(pickle_path, "wb") as f:
            pickle.dump(model, f)
        mlflow.log_artifact(pickle_path, artifact_path="model_pickle_backup")

    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)


def run_experiment(
    run_name,
    model,
    X_train,
    X_test,
    y_train,
    y_test,
    params,
    leaky,
    description
):
    with mlflow.start_run(run_name=run_name):
        
        # -----------------------
        # Params
        # -----------------------
        mlflow.log_param("dataset_path", DATA_PATH)
        mlflow.log_param("dataset_hash", DATASET_HASH)
        mlflow.log_param("dataset_version", DATASET_VERSION)
        mlflow.log_param("n_rows", df.shape[0])
        mlflow.log_param("n_raw_columns", df.shape[1])
        mlflow.log_param("n_features", X_train.shape[1])
        mlflow.log_param("target", TARGET)
        mlflow.log_param("leaky_features_used", leaky)
        mlflow.log_params(params)

        # -----------------------
        # Tags
        # -----------------------
        mlflow.set_tag("description", description)
        mlflow.set_tag("problem_type", "binary_classification")
        mlflow.set_tag("student", "sobhan_jabari")
        
        if leaky:
            mlflow.set_tag("valid_model", "false")
            mlflow.set_tag("reason", "uses future columns, leakage")
        else:
            mlflow.set_tag("valid_model", "true")

        # -----------------------
        # Train
        # -----------------------
        model.fit(X_train, y_train)

        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)

        # predict_proba برای AUC
        if hasattr(model, "predict_proba"):
            y_proba_train = model.predict_proba(X_train)[:, 1]
            y_proba_test = model.predict_proba(X_test)[:, 1]
        else:
            y_proba_train = y_pred_train
            y_proba_test = y_pred_test

        # -----------------------
        # Metrics
        # -----------------------
        metrics = {
            "train_accuracy": accuracy_score(y_train, y_pred_train),
            "test_accuracy": accuracy_score(y_test, y_pred_test),
            "train_precision": precision_score(y_train, y_pred_train, zero_division=0),
            "test_precision": precision_score(y_test, y_pred_test, zero_division=0),
            "train_recall": recall_score(y_train, y_pred_train, zero_division=0),
            "test_recall": recall_score(y_test, y_pred_test, zero_division=0),
            "train_f1": f1_score(y_train, y_pred_train, zero_division=0),
            "test_f1": f1_score(y_test, y_pred_test, zero_division=0),
            "train_auc": roc_auc_score(y_train, y_proba_train),
            "test_auc": roc_auc_score(y_test, y_proba_test),
        }

        metrics["auc_gap"] = metrics["train_auc"] - metrics["test_auc"]
        metrics["f1_gap"] = metrics["train_f1"] - metrics["test_f1"]

        mlflow.log_metrics(metrics)

        # -----------------------
        # Artifacts
        # -----------------------
        log_confusion_matrix_artifact(y_test, y_pred_test)

        # Feature importance برای مدل‌های tree-based
        if hasattr(model, "feature_importances_"):
            fi = pd.Series(model.feature_importances_, index=X_train.columns)
            fi = fi.sort_values(ascending=False)

            fi.to_csv("feature_importance.csv")
            mlflow.log_artifact("feature_importance.csv")

            print(f"\nTop 10 features for {run_name}:")
            display(fi.head(10))

        # -----------------------
        # Model artifact
        # -----------------------
        save_and_log_model_as_artifact(model, X_train, artifact_path="model")

        print("=" * 80)
        print(run_name)
        print("=" * 80)
        for k, v in metrics.items():
            print(f"{k}: {v:.4f}")

        return metrics

In [53]:
dummy_model = DummyClassifier(strategy="most_frequent")

metrics_dummy = run_experiment(
    run_name="v0_dummy_baseline_clean_fixed_artifact",
    model=dummy_model,
    X_train=X_train_clean,
    X_test=X_test_clean,
    y_train=y_train_clean,
    y_test=y_test_clean,
    params={
        "model_type": "DummyClassifier",
        "strategy": "most_frequent",
        "split": "random_stratified",
    },
    leaky=False,
    description="Baseline dummy model using only the majority class."
)

C:\Users\sobhan\AppData\Roaming\Python\Python312\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/06/02 16:25:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute a

v0_dummy_baseline_clean_fixed_artifact
train_accuracy: 0.7628
test_accuracy: 0.7629
train_precision: 0.7628
test_precision: 0.7629
train_recall: 1.0000
test_recall: 1.0000
train_f1: 0.8654
test_f1: 0.8655
train_auc: 0.5000
test_auc: 0.5000
auc_gap: 0.0000
f1_gap: -0.0001
🏃 View run v0_dummy_baseline_clean_fixed_artifact at: http://185.50.38.163:33014/#/experiments/45/runs/bf119000fff94be89abccb3995f72531
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/45


In [55]:
leaky_rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

metrics_leaky_rf = run_experiment(
    run_name="v1_leaky_random_forest_bad_model_fixed_artifact",
    model=leaky_rf_model,
    X_train=X_train_leaky,
    X_test=X_test_leaky,
    y_train=y_train_leaky,
    y_test=y_test_leaky,
    params={
        "model_type": "RandomForestClassifier",
        "n_estimators": 300,
        "max_depth": "None",
        "split": "random_stratified",
        "uses_future_features": True,
        "leaky_columns": ",".join(LEAKY_COLS),
    },
    leaky=True,
    description="Bad model intentionally using future availability columns to demonstrate data leakage."
)


Top 10 features for v1_leaky_random_forest_bad_model_fixed_artifact:


future_available_rate_30d      0.267266
future_available_days_30d      0.215991
available_rate_last_30d        0.157000
available_days_last_30d        0.126554
available_days_history         0.103651
available_rate_history         0.090434
listing_price                  0.008226
days_since_last_review         0.003828
beds                           0.003480
total_reviews_before_cutoff    0.002299
dtype: float64

C:\Users\sobhan\AppData\Roaming\Python\Python312\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/06/02 16:27:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute a

v1_leaky_random_forest_bad_model_fixed_artifact
train_accuracy: 1.0000
test_accuracy: 1.0000
train_precision: 1.0000
test_precision: 1.0000
train_recall: 1.0000
test_recall: 1.0000
train_f1: 1.0000
test_f1: 1.0000
train_auc: 1.0000
test_auc: 1.0000
auc_gap: 0.0000
f1_gap: 0.0000
🏃 View run v1_leaky_random_forest_bad_model_fixed_artifact at: http://185.50.38.163:33014/#/experiments/45/runs/ee2ad3325d684503bccfeb49c0d82218
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/45


## 12. Run 2 — clean logistic regression

Train your first clean real model.

Use only `X_clean`.

Required tags:

```text
leakage_status = clean
model_family = logistic_regression
```

In [ ]:
# TODO: build and log clean LogisticRegression.

raise NotImplementedError("Run and log v2_clean_logistic_regression.")

In [57]:
logreg_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]
)

metrics_logreg = run_experiment(
    run_name="v2_logistic_regression_clean_fixed_artifact",
    model=logreg_model,
    X_train=X_train_clean,
    X_test=X_test_clean,
    y_train=y_train_clean,
    y_test=y_test_clean,
    params={
        "model_type": "LogisticRegression",
        "class_weight": "balanced",
        "max_iter": 2000,
        "split": "random_stratified",
        "uses_future_features": False,
    },
    leaky=False,
    description="Clean logistic regression baseline without future leakage features."
)

C:\Users\sobhan\AppData\Roaming\Python\Python312\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/06/02 16:28:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute a

v2_logistic_regression_clean_fixed_artifact
train_accuracy: 0.9810
test_accuracy: 0.9785
train_precision: 0.9858
test_precision: 0.9820
train_recall: 0.9894
test_recall: 0.9900
train_f1: 0.9876
test_f1: 0.9860
train_auc: 0.9929
test_auc: 0.9904
auc_gap: 0.0025
f1_gap: 0.0016
🏃 View run v2_logistic_regression_clean_fixed_artifact at: http://185.50.38.163:33014/#/experiments/45/runs/2e9d540b3b6943f48389cabfcfd4644c
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/45


In [59]:
import os
import shutil
import tempfile
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np

from mlflow.models.signature import infer_signature

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


def log_confusion_matrix_artifact(y_true, y_pred, filename="confusion_matrix.txt"):
    cm = confusion_matrix(y_true, y_pred)

    with open(filename, "w", encoding="utf-8") as f:
        f.write("Confusion Matrix\n")
        f.write(str(cm))
        f.write("\n\nClassification Report\n")
        f.write(classification_report(y_true, y_pred, zero_division=0))

    mlflow.log_artifact(filename)


def log_sklearn_model_compatible(model, X_train, artifact_path="model"):
    """
    Compatible model logging for older MLflow servers.
    Avoids mlflow.sklearn.log_model because your server does not support /logged-models.
    """
    tmp_dir = tempfile.mkdtemp()
    local_model_path = os.path.join(tmp_dir, artifact_path)

    try:
        signature = infer_signature(X_train, model.predict(X_train))

        mlflow.sklearn.save_model(
            sk_model=model,
            path=local_model_path,
            signature=signature,
            input_example=X_train.head(3)
        )

        mlflow.log_artifacts(local_model_path, artifact_path=artifact_path)

    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)


def run_experiment(
    run_name,
    model,
    X_train,
    X_test,
    y_train,
    y_test,
    params,
    leaky,
    description
):
    with mlflow.start_run(run_name=run_name):

        mlflow.log_param("dataset_path", DATA_PATH)
        mlflow.log_param("dataset_hash", DATASET_HASH)
        mlflow.log_param("dataset_version", DATASET_VERSION)
        mlflow.log_param("n_rows", df.shape[0])
        mlflow.log_param("n_raw_columns", df.shape[1])
        mlflow.log_param("n_features", X_train.shape[1])
        mlflow.log_param("target", TARGET)
        mlflow.log_param("leaky_features_used", leaky)
        mlflow.log_params(params)

        mlflow.set_tag("description", description)
        mlflow.set_tag("problem_type", "binary_classification")
        mlflow.set_tag("student", "sobhan_jabari")

        if leaky:
            mlflow.set_tag("valid_model", "false")
            mlflow.set_tag("reason", "uses future columns, leakage")
        else:
            mlflow.set_tag("valid_model", "true")

        model.fit(X_train, y_train)

        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)

        if hasattr(model, "predict_proba"):
            y_proba_train = model.predict_proba(X_train)[:, 1]
            y_proba_test = model.predict_proba(X_test)[:, 1]
        else:
            y_proba_train = y_pred_train
            y_proba_test = y_pred_test

        metrics = {
            "train_accuracy": accuracy_score(y_train, y_pred_train),
            "test_accuracy": accuracy_score(y_test, y_pred_test),
            "train_precision": precision_score(y_train, y_pred_train, zero_division=0),
            "test_precision": precision_score(y_test, y_pred_test, zero_division=0),
            "train_recall": recall_score(y_train, y_pred_train, zero_division=0),
            "test_recall": recall_score(y_test, y_pred_test, zero_division=0),
            "train_f1": f1_score(y_train, y_pred_train, zero_division=0),
            "test_f1": f1_score(y_test, y_pred_test, zero_division=0),
            "train_auc": roc_auc_score(y_train, y_proba_train),
            "test_auc": roc_auc_score(y_test, y_proba_test),
        }

        metrics["auc_gap"] = metrics["train_auc"] - metrics["test_auc"]
        metrics["f1_gap"] = metrics["train_f1"] - metrics["test_f1"]

        mlflow.log_metrics(metrics)

        log_confusion_matrix_artifact(y_test, y_pred_test)

        if hasattr(model, "feature_importances_"):
            fi = pd.Series(model.feature_importances_, index=X_train.columns)
            fi = fi.sort_values(ascending=False)

            fi.to_csv("feature_importance.csv")
            mlflow.log_artifact("feature_importance.csv")

            print(f"\nTop 10 features for {run_name}:")
            display(fi.head(10))

        log_sklearn_model_compatible(model, X_train, artifact_path="model")

        print("=" * 80)
        print(run_name)
        print("=" * 80)
        for k, v in metrics.items():
            print(f"{k}: {v:.4f}")

        return metrics, model

## 13. Run 3 — class-weighted logistic regression

Train logistic regression with:

```python
class_weight="balanced"
```

Compare precision and recall against the previous clean logistic model.

In [61]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

run3_logreg_balanced = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]
)

metrics_run3, fitted_run3_model = run_experiment(
    run_name="run3_class_weighted_logistic_regression_clean",
    model=run3_logreg_balanced,
    X_train=X_train_clean,
    X_test=X_test_clean,
    y_train=y_train_clean,
    y_test=y_test_clean,
    params={
        "run_number": 3,
        "model_type": "LogisticRegression",
        "class_weight": "balanced",
        "max_iter": 2000,
        "split": "random_stratified",
        "uses_future_features": False,
    },
    leaky=False,
    description="Run 3: class-weighted logistic regression without future leakage features."
)

C:\Users\sobhan\AppData\Roaming\Python\Python312\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/06/02 16:43:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute a

run3_class_weighted_logistic_regression_clean
train_accuracy: 0.9810
test_accuracy: 0.9785
train_precision: 0.9858
test_precision: 0.9820
train_recall: 0.9894
test_recall: 0.9900
train_f1: 0.9876
test_f1: 0.9860
train_auc: 0.9929
test_auc: 0.9904
auc_gap: 0.0025
f1_gap: 0.0016
🏃 View run run3_class_weighted_logistic_regression_clean at: http://185.50.38.163:33014/#/experiments/45/runs/195e406b23b1498e86fd536955e921ad
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/45


In [ ]:
# TODO: build and log class-weighted LogisticRegression.

raise NotImplementedError("Run and log v3_balanced_logistic_regression.")

## 14. Run 4 — threshold tuning

Use a fitted probability model and test several decision thresholds.

Suggested thresholds:

```text
0.30, 0.40, 0.50, 0.60
```

You may log one run per threshold.

The goal is to see how precision/recall/f1 change when the threshold changes.

In [63]:
from sklearn.metrics import precision_recall_fscore_support

y_proba_test_run3 = fitted_run3_model.predict_proba(X_test_clean)[:, 1]

threshold_results = []

for threshold in np.arange(0.10, 0.91, 0.01):
    y_pred_thr = (y_proba_test_run3 >= threshold).astype(int)

    precision = precision_score(y_test_clean, y_pred_thr, zero_division=0)
    recall = recall_score(y_test_clean, y_pred_thr, zero_division=0)
    f1 = f1_score(y_test_clean, y_pred_thr, zero_division=0)
    accuracy = accuracy_score(y_test_clean, y_pred_thr)

    threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_df = pd.DataFrame(threshold_results)

display(
    threshold_df.sort_values("f1", ascending=False).head(10)
)

,threshold,accuracy,precision,recall,f1
60,0.70,0.980439,0.986267,0.988118,0.987192
61,0.71,0.980439,0.986875,0.987492,0.987183
46,0.56,0.979962,0.983841,0.989994,0.986908
45,0.55,0.979962,0.983841,0.989994,0.986908
44,0.54,0.979962,0.983841,0.989994,0.986908
55,0.65,0.979962,0.985047,0.988743,0.986891
57,0.67,0.979962,0.985047,0.988743,0.986891
56,0.66,0.979962,0.985047,0.988743,0.986891
59,0.69,0.979962,0.985652,0.988118,0.986883
58,0.68,0.979962,0.985652,0.988118,0.986883


In [65]:
best_threshold_row = threshold_df.sort_values("f1", ascending=False).iloc[0]

BEST_THRESHOLD = float(best_threshold_row["threshold"])
BEST_THRESHOLD_F1 = float(best_threshold_row["f1"])

print("Best threshold:", BEST_THRESHOLD)
print("Best F1:", BEST_THRESHOLD_F1)

Best threshold: 0.6999999999999996
Best F1: 0.9871915026554202


In [67]:
with mlflow.start_run(run_name="run4_threshold_tuning_logistic_regression_clean"):

    y_proba_train = fitted_run3_model.predict_proba(X_train_clean)[:, 1]
    y_proba_test = fitted_run3_model.predict_proba(X_test_clean)[:, 1]

    y_pred_train_thr = (y_proba_train >= BEST_THRESHOLD).astype(int)
    y_pred_test_thr = (y_proba_test >= BEST_THRESHOLD).astype(int)

    metrics_run4 = {
        "train_accuracy": accuracy_score(y_train_clean, y_pred_train_thr),
        "test_accuracy": accuracy_score(y_test_clean, y_pred_test_thr),
        "train_precision": precision_score(y_train_clean, y_pred_train_thr, zero_division=0),
        "test_precision": precision_score(y_test_clean, y_pred_test_thr, zero_division=0),
        "train_recall": recall_score(y_train_clean, y_pred_train_thr, zero_division=0),
        "test_recall": recall_score(y_test_clean, y_pred_test_thr, zero_division=0),
        "train_f1": f1_score(y_train_clean, y_pred_train_thr, zero_division=0),
        "test_f1": f1_score(y_test_clean, y_pred_test_thr, zero_division=0),
        "train_auc": roc_auc_score(y_train_clean, y_proba_train),
        "test_auc": roc_auc_score(y_test_clean, y_proba_test),
    }

    metrics_run4["auc_gap"] = metrics_run4["train_auc"] - metrics_run4["test_auc"]
    metrics_run4["f1_gap"] = metrics_run4["train_f1"] - metrics_run4["test_f1"]

    mlflow.log_param("dataset_path", DATA_PATH)
    mlflow.log_param("dataset_hash", DATASET_HASH)
    mlflow.log_param("dataset_version", DATASET_VERSION)
    mlflow.log_param("n_rows", df.shape[0])
    mlflow.log_param("n_raw_columns", df.shape[1])
    mlflow.log_param("n_features", X_train_clean.shape[1])
    mlflow.log_param("target", TARGET)
    mlflow.log_param("leaky_features_used", False)

    mlflow.log_param("run_number", 4)
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("base_model", "run3_class_weighted_logistic_regression_clean")
    mlflow.log_param("threshold", BEST_THRESHOLD)
    mlflow.log_param("threshold_selection_metric", "test_f1")
    mlflow.log_param("uses_future_features", False)

    mlflow.set_tag("description", "Run 4: threshold tuning on class-weighted logistic regression.")
    mlflow.set_tag("problem_type", "binary_classification")
    mlflow.set_tag("student", "sobhan_jabari")
    mlflow.set_tag("valid_model", "true")

    mlflow.log_metrics(metrics_run4)

    # ذخیره جدول thresholdها
    threshold_df.to_csv("threshold_tuning_results.csv", index=False)
    mlflow.log_artifact("threshold_tuning_results.csv")

    # confusion matrix با threshold جدید
    log_confusion_matrix_artifact(
        y_test_clean,
        y_pred_test_thr,
        filename="confusion_matrix_threshold_tuned.txt"
    )

    # ذخیره همان مدل Run 3 به عنوان artifact این Run هم
    log_sklearn_model_compatible(
        fitted_run3_model,
        X_train_clean,
        artifact_path="model"
    )

    print("=" * 80)
    print("run4_threshold_tuning_logistic_regression_clean")
    print("=" * 80)
    print("Best threshold:", BEST_THRESHOLD)
    for k, v in metrics_run4.items():
        print(f"{k}: {v:.4f}")

C:\Users\sobhan\AppData\Roaming\Python\Python312\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/06/02 16:45:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute a

run4_threshold_tuning_logistic_regression_clean
Best threshold: 0.6999999999999996
train_accuracy: 0.9818
test_accuracy: 0.9804
train_precision: 0.9893
test_precision: 0.9863
train_recall: 0.9867
test_recall: 0.9881
train_f1: 0.9880
test_f1: 0.9872
train_auc: 0.9929
test_auc: 0.9904
auc_gap: 0.0025
f1_gap: 0.0008
🏃 View run run4_threshold_tuning_logistic_regression_clean at: http://185.50.38.163:33014/#/experiments/45/runs/1c11053a6f0e429eb491091ed380b5fd
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/45


In [68]:
# TODO: log threshold-tuning runs.

raise NotImplementedError("Run and log threshold tuning experiments.")

NotImplementedError: Run and log threshold tuning experiments.

## 15. Run 5 — tree-based model

Train a `RandomForestClassifier`.

This compares a nonlinear model against logistic regression.

Log at least these parameters:

```text
n_estimators
max_depth
min_samples_leaf
class_weight
random_state
```

In [ ]:
# TODO: build and log RandomForestClassifier.

raise NotImplementedError("Run and log v5_random_forest.")

In [70]:
from sklearn.ensemble import RandomForestClassifier

run5_tree_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

metrics_run5, fitted_run5_model = run_experiment(
    run_name="run5_tree_based_random_forest_clean",
    model=run5_tree_model,
    X_train=X_train_clean,
    X_test=X_test_clean,
    y_train=y_train_clean,
    y_test=y_test_clean,
    params={
        "run_number": 5,
        "model_type": "RandomForestClassifier",
        "n_estimators": 300,
        "max_depth": 12,
        "min_samples_leaf": 10,
        "class_weight": "balanced",
        "split": "random_stratified",
        "uses_future_features": False,
    },
    leaky=False,
    description="Run 5: clean tree-based Random Forest model without future leakage features."
)


Top 10 features for run5_tree_based_random_forest_clean:


available_rate_last_30d           0.301869
available_days_last_30d           0.218137
available_rate_history            0.205798
available_days_history            0.183371
listing_price                     0.023558
beds                              0.012129
days_since_last_review            0.011279
unique_reviewers_before_cutoff    0.004460
total_reviews_before_cutoff       0.004221
host_listing_count                0.003889
dtype: float64

C:\Users\sobhan\AppData\Roaming\Python\Python312\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/06/02 16:46:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute a

run5_tree_based_random_forest_clean
train_accuracy: 0.9832
test_accuracy: 0.9809
train_precision: 0.9914
test_precision: 0.9863
train_recall: 0.9866
test_recall: 0.9887
train_f1: 0.9889
test_f1: 0.9875
train_auc: 0.9990
test_auc: 0.9947
auc_gap: 0.0043
f1_gap: 0.0014
🏃 View run run5_tree_based_random_forest_clean at: http://185.50.38.163:33014/#/experiments/45/runs/78b941ee9925435dbde9df7329d2a7cc
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/45


In [74]:
import os
import mlflow

MLFLOW_URI = "http://185.50.38.163:33014"
EXPERIMENT_NAME = "qbc12_hw02_sobhan_jabari"

os.environ["MLFLOW_TRACKING_USERNAME"] = "sobhan_jabari"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "iovNbOmVm0anGUMT"

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", mlflow.get_experiment_by_name(EXPERIMENT_NAME))

Tracking URI: http://185.50.38.163:33014
Experiment: <Experiment: artifact_location='mlflow-artifacts:/45', creation_time=1780397019709, experiment_id='45', last_update_time=1780397019709, lifecycle_stage='active', name='qbc12_hw02_sobhan_jabari', tags={}, workspace='default'>


In [76]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.test_auc DESC"]
)

cols = [
    "run_id",
    "tags.mlflow.runName",
    "tags.valid_model",
    "metrics.test_auc",
    "metrics.test_f1",
    "metrics.test_accuracy",
    "metrics.test_precision",
    "metrics.test_recall",
    "metrics.train_auc",
    "metrics.auc_gap",
    "params.run_number",
    "params.model_type",
    "params.threshold",
    "params.leaky_features_used",
]

available_cols = [c for c in cols if c in runs.columns]

comparison = runs[available_cols].copy()
display(comparison)

,run_id,tags.mlflow.runName,tags.valid_model,metrics.test_auc,metrics.test_f1,metrics.test_accuracy,metrics.test_precision,metrics.test_recall,metrics.train_auc,metrics.auc_gap,params.run_number,params.model_type,params.threshold,params.leaky_features_used
0,ee2ad3325d684503bccfeb49c0d82218,v1_leaky_random_forest_bad_model_fixed_artifact,false,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,None,RandomForestClassifier,None,True
1,1855788d93c4413fa38e5a735797cc0b,v1_leaky_random_forest_bad_model,false,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,None,RandomForestClassifier,None,True
2,78b941ee9925435dbde9df7329d2a7cc,run5_tree_based_random_forest_clean,true,0.994711,0.987508,0.980916,0.986276,0.988743,0.998969,0.004258,5,RandomForestClassifier,None,False
3,1c11053a6f0e429eb491091ed380b5fd,run4_threshold_tuning_logistic_regression_clean,true,0.990391,0.987192,0.980439,0.986267,0.988118,0.992912,0.002521,4,LogisticRegression,0.6999999999999996,False
4,195e406b23b1498e86fd536955e921ad,run3_class_weighted_logistic_regression_clean,true,0.990391,0.985986,0.978531,0.982010,0.989994,0.992912,0.002521,3,LogisticRegression,None,False
5,2e9d540b3b6943f48389cabfcfd4644c,v2_logistic_regression_clean_fixed_artifact,true,0.990391,0.985986,0.978531,0.982010,0.989994,0.992912,0.002521,None,LogisticRegression,None,False
6,bf119000fff94be89abccb3995f72531,v0_dummy_baseline_clean_fixed_artifact,true,0.500000,0.865494,0.762882,0.762882,1.000000,0.500000,0.000000,None,DummyClassifier,None,False
7,78f4e531dc4a4e2ba62f40eecb3c9974,v0_dummy_baseline_clean,true,0.500000,0.865494,0.762882,0.762882,1.000000,0.500000,0.000000,None,DummyClassifier,None,False
8,2d580bfa3f5742aba9f24af5155fbbbd,v0_dummy_baseline_clean,true,0.500000,0.865494,0.762882,0.762882,1.000000,0.500000,0.000000,None,DummyClassifier,None,False
9,231451c6a7544b8bbf5fcf0465ad86ba,v0_dummy_baseline_clean,true,0.500000,0.865494,0.762882,0.762882,1.000000,0.500000,0.000000,None,DummyClassifier,None,False


## 16. Compare MLflow runs

Use `mlflow.search_runs` to retrieve your experiment runs.

Compare at least:

```text
run name
leakage status
model family
accuracy
precision
recall
f1
roc_auc
```

Do not select a leaky run as final candidate.

In [ ]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

# TODO: retrieve MLflow runs for this experiment and create a comparison table.
raise NotImplementedError("Create runs comparison table.")

comparison_df

## 17. Select final candidate

Pick the best **clean** run.

Do not choose the leaky run.

Selection should be based on:

- f1
- roc_auc
- precision/recall tradeoff
- no leakage
- full preprocessing Pipeline logged

Write a short explanation.

In [78]:
valid_runs = runs[runs["tags.valid_model"] == "true"].copy()

valid_cols = [
    "run_id",
    "tags.mlflow.runName",
    "metrics.test_auc",
    "metrics.test_f1",
    "metrics.test_accuracy",
    "metrics.test_precision",
    "metrics.test_recall",
    "metrics.auc_gap",
    "params.run_number",
    "params.model_type",
    "params.threshold",
]

valid_cols = [c for c in valid_cols if c in valid_runs.columns]

valid_comparison = valid_runs[valid_cols].sort_values(
    by="metrics.test_auc",
    ascending=False
)

display(valid_comparison)

,run_id,tags.mlflow.runName,metrics.test_auc,metrics.test_f1,metrics.test_accuracy,metrics.test_precision,metrics.test_recall,metrics.auc_gap,params.run_number,params.model_type,params.threshold
2,78b941ee9925435dbde9df7329d2a7cc,run5_tree_based_random_forest_clean,0.994711,0.987508,0.980916,0.986276,0.988743,0.004258,5,RandomForestClassifier,None
3,1c11053a6f0e429eb491091ed380b5fd,run4_threshold_tuning_logistic_regression_clean,0.990391,0.987192,0.980439,0.986267,0.988118,0.002521,4,LogisticRegression,0.6999999999999996
4,195e406b23b1498e86fd536955e921ad,run3_class_weighted_logistic_regression_clean,0.990391,0.985986,0.978531,0.982010,0.989994,0.002521,3,LogisticRegression,None
5,2e9d540b3b6943f48389cabfcfd4644c,v2_logistic_regression_clean_fixed_artifact,0.990391,0.985986,0.978531,0.982010,0.989994,0.002521,None,LogisticRegression,None
6,bf119000fff94be89abccb3995f72531,v0_dummy_baseline_clean_fixed_artifact,0.500000,0.865494,0.762882,0.762882,1.000000,0.000000,None,DummyClassifier,None
7,78f4e531dc4a4e2ba62f40eecb3c9974,v0_dummy_baseline_clean,0.500000,0.865494,0.762882,0.762882,1.000000,0.000000,None,DummyClassifier,None
8,2d580bfa3f5742aba9f24af5155fbbbd,v0_dummy_baseline_clean,0.500000,0.865494,0.762882,0.762882,1.000000,0.000000,None,DummyClassifier,None
9,231451c6a7544b8bbf5fcf0465ad86ba,v0_dummy_baseline_clean,0.500000,0.865494,0.762882,0.762882,1.000000,0.000000,None,DummyClassifier,None


In [ ]:
# TODO: set BEST_RUN_ID to the selected clean run ID.
BEST_RUN_ID = None

if BEST_RUN_ID is None:
    raise ValueError("Set BEST_RUN_ID to your selected clean MLflow run ID.")

client.set_tag(BEST_RUN_ID, "selected_for_serving", "true")
client.set_tag(BEST_RUN_ID, "production_candidate", "true")

print("Selected best run:", BEST_RUN_ID)

In [80]:
best_valid_run = valid_comparison.sort_values(
    by="metrics.test_auc",
    ascending=False
).iloc[0]

BEST_RUN_ID = best_valid_run["run_id"]
BEST_RUN_NAME = best_valid_run["tags.mlflow.runName"]
BEST_TEST_AUC = best_valid_run["metrics.test_auc"]
BEST_TEST_F1 = best_valid_run["metrics.test_f1"]

print("Best valid run ID:", BEST_RUN_ID)
print("Best valid run name:", BEST_RUN_NAME)
print("Best test AUC:", BEST_TEST_AUC)
print("Best test F1:", BEST_TEST_F1)

Best valid run ID: 78b941ee9925435dbde9df7329d2a7cc
Best valid run name: run5_tree_based_random_forest_clean
Best test AUC: 0.9947112317431795
Best test F1: 0.9875078076202374


## Final explanation

Write 3–6 sentences:

- Which run did you select?
- Why did you select it?
- Why did you reject the leaky run?
- What would you try next?

In [84]:
# TODO: replace this text.
final_explanation = """
I selected Run 5 — run5_tree_based_random_forest_clean as the final model because it achieved the best valid performance among the non-future-leaky runs, with the highest test AUC and F1 score and only a small train-test gap.

Compared with the logistic regression runs, the RandomForest model captured non-linear relationships better and produced slightly stronger evaluation metrics.

I rejected the leaky run because it used future availability information, so its performance would be unrealistically optimistic and would not generalize to a real prediction setting.

Feature importance analysis also showed that Run 5 relied heavily on historical availability proxy features, so as a next step I would run a stricter experiment removing these availability proxy variables and compare the performance again.

I would also try threshold tuning for the tree-based model and possibly evaluate models using a time-based split instead of only a random stratified split.
"""

print(final_explanation)


I selected Run 5 — run5_tree_based_random_forest_clean as the final model because it achieved the best valid performance among the non-future-leaky runs, with the highest test AUC and F1 score and only a small train-test gap.

Compared with the logistic regression runs, the RandomForest model captured non-linear relationships better and produced slightly stronger evaluation metrics.

I rejected the leaky run because it used future availability information, so its performance would be unrealistically optimistic and would not generalize to a real prediction setting.

Feature importance analysis also showed that Run 5 relied heavily on historical availability proxy features, so as a next step I would run a stricter experiment removing these availability proxy variables and compare the performance again.

I would also try threshold tuning for the tree-based model and possibly evaluate models using a time-based split instead of only a random stratified split.

